# Task 2 ? Exploratory Data Analysis (EDA)

## Objective
Analyze the HR Employee Attrition dataset using Pandas.

**Tasks covered:**
1. Summary statistics
2. Department-wise group analysis using `groupby().agg()`
3. Top 3 numerical correlations
4. Business questions with numerical answers
5. Department budget merge using `pd.merge()`
6. Bonus: Department and Gender pivot table

**Note:** Budget values used in this exercise are illustrative examples, not actual company budgets.


In [ ]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "WA_Fn-UseC_-HR-Employee-Attrition.csv"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PATH)

print("DATASET OVERVIEW")
print("Shape:", df.shape)
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))
df.head()


## 1. Summary Statistics

Calculate mean, median, minimum, maximum and standard deviation for key numerical columns.


In [ ]:
numeric_columns = [
    "Age", "MonthlyIncome", "YearsAtCompany", "TotalWorkingYears"
]

summary_stats = df[numeric_columns].agg(
    ["mean", "median", "min", "max", "std"]
)

print("SUMMARY STATISTICS")
display(summary_stats.round(2))


## 2. Department-wise Analysis

Use `groupby().agg()` to compare employee count, average age, average salary, experience and attrition across departments.


In [ ]:
department_analysis = df.groupby("Department").agg(
    Employee_Count=("EmployeeNumber", "count"),
    Average_Age=("Age", "mean"),
    Average_Salary=("MonthlyIncome", "mean"),
    Average_Experience=("TotalWorkingYears", "mean"),
    Attrition_Count=("Attrition", lambda x: (x == "Yes").sum())
).round(2)

department_analysis["Attrition_Rate"] = (
    department_analysis["Attrition_Count"] /
    department_analysis["Employee_Count"] * 100
).round(2)

display(department_analysis)

department_analysis.to_csv(
    OUTPUT_DIR / "department_analysis.csv"
)


## 3. Top 3 Correlations

Calculate correlations between numerical columns and identify the three strongest relationships.

Correlation indicates association; it does not prove causation.


In [ ]:
excluded_columns = [
    "EmployeeNumber", "EmployeeCount", "StandardHours"
]

corr_columns = [
    col for col in df.select_dtypes(include="number").columns
    if col not in excluded_columns
]

corr = df[corr_columns].corr()

mask = pd.DataFrame(
    [[i < j for j in range(len(corr.columns))]
     for i in range(len(corr.columns))],
    index=corr.index,
    columns=corr.columns
)

top_correlations = corr.where(mask).stack().reset_index()
top_correlations.columns = ["Column 1", "Column 2", "Correlation"]
top_correlations["Absolute Correlation"] = (
    top_correlations["Correlation"].abs()
)

top_3 = top_correlations.sort_values(
    "Absolute Correlation", ascending=False
).head(3)

print("TOP 3 CORRELATIONS")
display(top_3)

top_3.to_csv(
    OUTPUT_DIR / "top_3_correlations.csv",
    index=False
)


## 4. Business Questions and Numerical Answers

Answer five business questions using the dataset.


In [ ]:
total_employees = len(df)
attrition_count = (df["Attrition"] == "Yes").sum()
overall_attrition_rate = attrition_count / total_employees * 100

print("Q1. What is the overall attrition rate?")
print(f"Answer: {overall_attrition_rate:.2f}% ({attrition_count} out of {total_employees} employees)")

department_attrition = df.assign(
    AttritionFlag=(df["Attrition"] == "Yes").astype(int)
).groupby("Department")["AttritionFlag"].mean() * 100

print("\nQ2. Which department has the highest attrition rate?")
print(f"Answer: {department_attrition.idxmax()} ({department_attrition.max():.2f}%)")

overtime_attrition = df.assign(
    AttritionFlag=(df["Attrition"] == "Yes").astype(int)
).groupby("OverTime")["AttritionFlag"].mean() * 100

print("\nQ3. What is the attrition rate for employees working overtime?")
print(f"Answer: {overtime_attrition.get('Yes', 0):.2f}%")

print("\nQ4. Which department has the highest average monthly income?")
print(f"Answer: {department_analysis['Average_Salary'].idxmax()}")

print("\nQ5. What is the average employee age?")
print(f"Answer: {df['Age'].mean():.2f} years")


## 5. Merge Department Budget Data

Create a small illustrative department budget table and merge it with the department analysis using `pd.merge()`.


In [ ]:
budget_df = pd.DataFrame({
    "Department": [
        "Human Resources",
        "Research & Development",
        "Sales"
    ],
    "Illustrative_Budget": [500000, 3000000, 2000000]
})

merged_df = pd.merge(
    department_analysis.reset_index(),
    budget_df,
    on="Department",
    how="left"
)

display(merged_df)

merged_df.to_csv(
    OUTPUT_DIR / "department_budget_analysis.csv",
    index=False
)


## 6. Bonus ? Department and Gender Pivot Table

Use `pd.pivot_table()` to compare employee counts by department and gender.


In [ ]:
pivot = pd.pivot_table(
    df,
    index="Department",
    columns="Gender",
    values="EmployeeNumber",
    aggfunc="count",
    fill_value=0
)

display(pivot)

pivot.to_csv(
    OUTPUT_DIR / "department_gender_pivot.csv"
)


## Conclusion

The analysis covers summary statistics, department comparisons, correlations, business questions, department budget merging and a department-gender pivot table.

All generated CSV reports are saved in the `output` folder.

**Important:** The illustrative department budgets are sample figures created for this exercise.
